# 🏙️ Smart City Traffic & Accident Analytics
**MetroCity — Intelligent Traffic Management System**

---

### Pipeline
```
Raw CSVs  →  ETL (Extract → Transform → Load)  →  MySQL Data Warehouse  →  Power BI
```

### Datasets
| File | Rows | Columns |
|------|------|---------|
| `road_traffic_sensor_data.csv` | 300 | Sensor_ID, Location, Date_Time, Vehicle_Count, Average_Speed, Congestion_Level |
| `traffic_accident_data.csv` | 5,000 | Accident_ID, Date_Time, Location, Weather_Condition, Road_Condition, Vehicle_Type, Accident_Severity, Number_of_Vehicles, Casualties, Traffic_Density |

> **Assumption:** All traffic sensor statuses are **Active**.

### MySQL Schema (smart_city_dw)
| Object | Type | Purpose |
|--------|------|---------|
| `dim_location` | Dimension | Location master with zone classification |
| `dim_date` | Dimension | Full datetime attributes + sort keys for Power BI |
| `dim_vehicle` | Dimension | Vehicle type + class |
| `dim_weather` | Dimension | Weather condition + adverse flag |
| `fact_traffic` | Fact | Enriched sensor readings with KPIs |
| `fact_accident` | Fact | Enriched accident records with KPIs + denormalised columns |
| `ml_accident_risk` | ML Output | Per-accident predicted risk probability and tier |
| `vw_location_summary` | View | Combined traffic + accident KPIs per location (incl. accident rate per 1000 vehicles) |
| `vw_hourly_profile` | View | 24-hour traffic + accident profile |
| `vw_monthly_trend` | View | Year-month accident and traffic trend |
| `vw_congestion_hotspots` | View | Peak-hour congestion by location × time period |
| `vw_weather_risk` | View | Weather × road condition risk matrix |
| `vw_severity_by_location` | View | Accident severity breakdown per location (stacked bar ready) |
| `vw_road_condition_risk` | View | Standalone road condition risk breakdown |
| `vw_ml_location_risk` | View | Aggregated ML predicted risk per location (map visual ready) |

---
## ⚙️ Section 1 — Imports & MySQL Configuration

In [32]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

import mysql.connector
from mysql.connector import Error

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, f1_score

print(' All libraries imported.')

 All libraries imported.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MySQL CONNECTION SETTINGS 
# ─────────────────────────────────────────────────────────────────────────────
MYSQL_CONFIG = {
    'host'    : 'localhost',
    'port'    : 3306,
    'user'    : 'root',
    'password': 'Monika#80',  
    'database': 'smart_city_dw'
}

SENSOR_FILE   = 'road_traffic_sensor_data.csv'
ACCIDENT_FILE = 'traffic_accident_data.csv'

# Helper: get connection 
def get_conn(include_db=True):
    cfg = MYSQL_CONFIG.copy()
    if not include_db:
        cfg.pop('database')
    return mysql.connector.connect(**cfg)

#  Helper: execute SQL (single or bulk) 
def exec_sql(cur, sql, vals=None, many=False):
    try:
        if many and vals: cur.executemany(sql, vals)
        elif vals:        cur.execute(sql, vals)
        else:             cur.execute(sql)
    except Error as e:
        print(f'  SQL Error: {e}'); raise

#  Helper: SELECT → DataFrame 
def qdf(sql, label=''):
    conn = get_conn()
    df   = pd.read_sql(sql, conn)
    conn.close()
    if label:
        print(f'\n── {label} ──')
        display(df)
    return df

print('Config and helpers ready.')

Config and helpers ready.


---
## 🗄️ Section 2 — Create MySQL Database & Tables

In [34]:
# ── Create database ───────────────────────────────────────────────────────────
conn = get_conn(include_db=False)
cur  = conn.cursor()
cur.execute('CREATE DATABASE IF NOT EXISTS smart_city_dw CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci')
conn.close()
print('Database `smart_city_dw` ready.')

Database `smart_city_dw` ready.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DDL — Drop in FK-safe order then recreate all tables
# ─────────────────────────────────────────────────────────────────────────────
STMTS = [

    # Drop order: child tables first
    'DROP TABLE IF EXISTS ml_accident_risk',
    'DROP TABLE IF EXISTS fact_accident',
    'DROP TABLE IF EXISTS fact_traffic',
    'DROP TABLE IF EXISTS dim_weather',
    'DROP TABLE IF EXISTS dim_vehicle',
    'DROP TABLE IF EXISTS dim_date',
    'DROP TABLE IF EXISTS dim_location',

    # ── DIMENSION: dim_location ───────────────────────────────────────────────
    # Master location table. Referenced by both fact tables.
    # zone_type enables grouping (Urban Core / Highway / Residential etc.)
    '''
    CREATE TABLE dim_location (
        location_id   INT AUTO_INCREMENT PRIMARY KEY,
        location_name VARCHAR(100) NOT NULL UNIQUE,
        zone_type     VARCHAR(50)  NOT NULL
    )
    ''',

    #  DIMENSION: dim_date 
    # One row per unique timestamp from the data.
    # Includes quarter, day_of_week_num and month sort columns so Power BI
    # can sort month_name and day_name labels correctly without a custom measure.
    # FIX vs previous: date_only stored as DATE (not VARCHAR string).
    '''
    CREATE TABLE dim_date (
        date_id          INT         AUTO_INCREMENT PRIMARY KEY,
        full_datetime    DATETIME    NOT NULL UNIQUE,
        date_only        DATE        NOT NULL,
        year             SMALLINT    NOT NULL,
        quarter          TINYINT     NOT NULL,
        month            TINYINT     NOT NULL,
        month_name       VARCHAR(15) NOT NULL,
        month_sort       TINYINT     NOT NULL,
        week             TINYINT     NOT NULL,
        day_name         VARCHAR(12) NOT NULL,
        day_of_week_num  TINYINT     NOT NULL,
        hour             TINYINT     NOT NULL,
        time_of_day      VARCHAR(20) NOT NULL,
        is_peak_hour     TINYINT(1)  NOT NULL,
        is_weekend       TINYINT(1)  NOT NULL
    )
    ''',

    # ── DIMENSION: dim_vehicle ────────────────────────────────────────────────
    # Vehicle type lookup. vehicle_class groups types into broader categories
    # (Heavy Motor, Light Motor, Two-Wheeler, Non-Motorised) for Power BI grouping.
    '''
    CREATE TABLE dim_vehicle (
        vehicle_id    INT AUTO_INCREMENT PRIMARY KEY,
        vehicle_type  VARCHAR(30) NOT NULL UNIQUE,
        vehicle_class VARCHAR(25) NOT NULL
    )
    ''',

    # ── DIMENSION: dim_weather ────────────────────────────────────────────────
    # Weather condition lookup with is_adverse flag.
    # is_adverse = 1 for Fog, Snow, Storm (hazardous visibility/traction).
    '''
    CREATE TABLE dim_weather (
        weather_id        INT AUTO_INCREMENT PRIMARY KEY,
        weather_condition VARCHAR(30) NOT NULL UNIQUE,
        is_adverse        TINYINT(1)  NOT NULL
    )
    ''',

    # ── FACT: fact_traffic ────────────────────────────────────────────────────
    # One row per sensor reading (300 rows).
    # Engineered KPIs:
    #   congestion_index : 0–1 composite = 0.6×(vol/max_vol) + 0.4×(1−spd/max_spd)
    #                      Higher = more congested. Anchored to dataset max.
    #   congestion_level_num : ordinal 1/2/3 for Power BI sorting and measures
    #   speed_category   : human-readable band with range in label for clarity
    #   volume_band      : human-readable vehicle count band with range in label
    '''
    CREATE TABLE fact_traffic (
        traffic_id            INT         AUTO_INCREMENT PRIMARY KEY,
        sensor_id             VARCHAR(20) NOT NULL,
        location_id           INT         NOT NULL,
        date_id               INT         NOT NULL,
        vehicle_count         INT         NOT NULL,
        average_speed         INT         NOT NULL,
        congestion_level      VARCHAR(20) NOT NULL,
        congestion_level_num  TINYINT     NOT NULL,
        congestion_index      FLOAT       NOT NULL,
        speed_category        VARCHAR(25) NOT NULL,
        volume_band           VARCHAR(25) NOT NULL,
        sensor_status         VARCHAR(20) NOT NULL DEFAULT 'Active',
        FOREIGN KEY (location_id) REFERENCES dim_location(location_id),
        FOREIGN KEY (date_id)     REFERENCES dim_date(date_id)
    )
    ''',

    # ── FACT: fact_accident ───────────────────────────────────────────────────
    # One row per accident (5,000 rows).
    # DESIGN DECISION: weather_condition and vehicle_type are kept as denormalised
    # columns IN ADDITION to FKs. This lets Power BI slicers and filters work on
    # fact_accident directly without always requiring a dim JOIN.
    #
    # Engineered KPIs:
    #   severity_score     : ordinal Minor=1, Moderate=2, Severe=3, Fatal=4
    #   risk_score         : severity × log1p(casualties+1) × vehicles
    #                        +1 inside log prevents 120 Fatal/0-casualty accidents
    #                        from scoring zero (log1p(0)=0 would erase severity signal)
    #   hazard_level       : Low / Moderate / High / Critical Hazard band
    #   adverse_weather    : 1 if Fog, Snow, or Storm
    #   hazardous_road     : 1 if Icy or Under Construction
    #   is_fatal_or_severe : binary ML target label
    '''
    CREATE TABLE fact_accident (
        accident_pk          INT         AUTO_INCREMENT PRIMARY KEY,
        accident_id          VARCHAR(20) NOT NULL UNIQUE,
        location_id          INT         NOT NULL,
        date_id              INT         NOT NULL,
        vehicle_id           INT         NOT NULL,
        weather_id           INT         NOT NULL,
        vehicle_type         VARCHAR(30) NOT NULL,
        weather_condition    VARCHAR(30) NOT NULL,
        road_condition       VARCHAR(30) NOT NULL,
        accident_severity    VARCHAR(20) NOT NULL,
        number_of_vehicles   INT         NOT NULL,
        casualties           INT         NOT NULL,
        traffic_density      VARCHAR(20) NOT NULL,
        severity_score       TINYINT     NOT NULL,
        risk_score           FLOAT       NOT NULL,
        hazard_level         VARCHAR(25) NOT NULL,
        adverse_weather      TINYINT(1)  NOT NULL,
        hazardous_road       TINYINT(1)  NOT NULL,
        is_peak_hour         TINYINT(1)  NOT NULL,
        is_fatal_or_severe   TINYINT(1)  NOT NULL,
        traffic_density_num  TINYINT     NOT NULL,
        FOREIGN KEY (location_id) REFERENCES dim_location(location_id),
        FOREIGN KEY (date_id)     REFERENCES dim_date(date_id),
        FOREIGN KEY (vehicle_id)  REFERENCES dim_vehicle(vehicle_id),
        FOREIGN KEY (weather_id)  REFERENCES dim_weather(weather_id)
    )
    ''',

    # ── ML OUTPUT: ml_accident_risk ───────────────────────────────────────────
    # One row per accident with predicted probability of Fatal/Severe outcome.
    # location_id FK added so Power BI can join directly to dim_location.
    # Powers the predictive risk map and risk tier donut chart.
    '''
    CREATE TABLE ml_accident_risk (
        accident_id            VARCHAR(20)  NOT NULL PRIMARY KEY,
        location_id            INT          NOT NULL,
        location_name          VARCHAR(100) NOT NULL,
        predicted_risk_prob    FLOAT        NOT NULL,
        predicted_fatal_severe TINYINT(1)   NOT NULL,
        risk_tier              VARCHAR(20)  NOT NULL,
        actual_fatal_severe    TINYINT(1)   NOT NULL,
        FOREIGN KEY (location_id) REFERENCES dim_location(location_id)
    )
    '''
]

conn = get_conn()
cur  = conn.cursor()
for s in STMTS:
    exec_sql(cur, s.strip())
conn.commit(); conn.close()

print(' Tables created:')
for t in ['dim_location','dim_date','dim_vehicle','dim_weather',
        'fact_traffic','fact_accident','ml_accident_risk']:
    print(f'   • {t}')

 Tables created:
   • dim_location
   • dim_date
   • dim_vehicle
   • dim_weather
   • fact_traffic
   • fact_accident
   • ml_accident_risk


---
## 📥 Section 3 — ETL: Extract (Load Raw Data)

In [ ]:
raw_sensor   = pd.read_csv(SENSOR_FILE)
raw_accident = pd.read_csv(ACCIDENT_FILE)

print('=' * 52)
print('EXTRACT — Raw Data Loaded')
print('=' * 52)
print(f'  Sensor   : {raw_sensor.shape[0]:>5,} rows × {raw_sensor.shape[1]} columns')
print(f'  Accident : {raw_accident.shape[0]:>5,} rows × {raw_accident.shape[1]} columns')

print('\n── Sensor Preview ──')
display(raw_sensor.head(3))
print('\n── Accident Preview ──')
display(raw_accident.head(3))

# ── Data Quality Audit ───────────────────────────────────────────────────────
print('\n── Null / Duplicate Audit ──')
print(f'  Nulls  — Sensor  : {raw_sensor.isnull().sum().sum()}')
print(f'  Nulls  — Accident: {raw_accident.isnull().sum().sum()}')
print(f'  Dupes  — Sensor  : {raw_sensor.duplicated().sum()}')
print(f'  Dupes  — Accident: {raw_accident.duplicated().sum()}')

print('\n── Unique Values per Categorical Column ──')
for col in raw_sensor.select_dtypes('object').columns:
    print(f'  Sensor  [{col}]: {raw_sensor[col].unique().tolist()}')
for col in raw_accident.select_dtypes('object').columns:
    print(f'  Accident[{col}]: {raw_accident[col].unique().tolist()}')

# ── Date range & important note about zero-casualty accidents ────────────────
raw_sensor_dt   = pd.to_datetime(raw_sensor['Date_Time'])
raw_accident_dt = pd.to_datetime(raw_accident['Date_Time'])
print(f'\n── Date Ranges ──')
print(f'  Sensor   : {raw_sensor_dt.min()}  →  {raw_sensor_dt.max()}')
print(f'  Accident : {raw_accident_dt.min()}  →  {raw_accident_dt.max()}')

zero_cas_fatal = ((raw_accident['Casualties']==0) &
                (raw_accident['Accident_Severity']=='Fatal')).sum()
print(f'\n  Zero-casualty Fatal accidents: {zero_cas_fatal}')
print('   → Risk Score uses log1p(casualties+1) to prevent these scoring 0.')
print('\n Extract complete — data is clean.')

EXTRACT — Raw Data Loaded
  Sensor   :   300 rows × 6 columns
  Accident : 5,000 rows × 10 columns

── Sensor Preview ──


,Sensor_ID,Location,Date_Time,Vehicle_Count,Average_Speed,Congestion_Level
0,a36f14,Downtown,2024-01-01 00:00:00,90,59,High
1,256a39,Downtown,2024-01-01 01:00:00,324,73,Moderate
2,a8a01f,Highway,2024-01-01 02:00:00,61,56,Moderate



── Accident Preview ──


,Accident_ID,Date_Time,Location,Weather_Condition,Road_Condition,Vehicle_Type,Accident_Severity,Number_of_Vehicles,Casualties,Traffic_Density
0,d02b3836,2024-01-01 00:00:00,Suburbs,Fog,Icy,Bus,Minor,2,6,Low
1,694c8577,2024-01-01 01:00:00,Downtown,Clear,Icy,Bus,Minor,2,2,Low
2,e0ed5c55,2024-01-01 02:00:00,Residential Zone,Snow,Dry,Motorcycle,Moderate,4,5,Moderate



── Null / Duplicate Audit ──
  Nulls  — Sensor  : 0
  Nulls  — Accident: 0
  Dupes  — Sensor  : 0
  Dupes  — Accident: 0

── Unique Values per Categorical Column ──
  Sensor  [Sensor_ID]: ['a36f14', '256a39', 'a8a01f', '3036e3', '2b45b1', '6204a2', '10cf13', '839bb8', '01274a', 'd0bc93', '333284', '0efa85', 'f2f551', 'f615a6', '989431', '87a3d3', 'e25a65', 'aac3db', '0c2b41', '0b9994', 'd08e95', '8b2539', '0657f4', 'bbf5ed', 'b2be52', '427ae9', '952d1a', 'd6931e', '37480a', '7bd9bd', '385ce7', '1941fe', '39fa33', '11ca6a', 'ea04d7', 'a58aa1', 'ca1ce3', '796f7d', '8cf7eb', '4a79f7', '434330', 'eaf3b6', '53ef76', '2735d9', '19c517', '75111a', '91f13e', '89a290', '1c20b7', '4f4463', '543c21', '0a17f2', '02b8dd', 'a5848a', 'bc5b3d', '3310a8', 'd6f7c2', 'f66341', 'e0b797', '1733ef', 'f479af', '5c13e9', '0555d9', '9eac86', 'e29033', '283c14', 'b87fe7', '738d77', 'e55908', '239899', 'd53e85', 'a3e5cc', '865c10', '4596db', '51db73', 'bb3017', '40f744', '0ac21c', '993101', '4478b2', 'e18116', 

---
## 🔧 Section 4 — ETL: Transform (Feature Engineering)

In [ ]:
# ── Classification helper functions ─────────────────────────────────────────

def time_of_day(h):
    """Classify hour into a named traffic period."""
    if   7 <= h <= 10:      return 'Morning Peak'
    elif 11 <= h <= 13:     return 'Midday'
    elif 17 <= h <= 20:     return 'Evening Peak'
    elif h >= 21 or h <= 5: return 'Off-Peak Night'
    else:                   return 'Off-Peak Day'

def speed_cat(s):
    if   s < 20: return 'Very Slow (<20 kmh)'
    elif s < 40: return 'Slow (20-39 kmh)'
    elif s < 60: return 'Moderate (40-59 kmh)'
    elif s < 80: return 'Fast (60-79 kmh)'
    else:        return 'Very Fast (80+ kmh)'

def volume_band(v):
    if   v <= 100: return 'Very Low (0-100)'
    elif v <= 200: return 'Low (101-200)'
    elif v <= 300: return 'Medium (201-300)'
    elif v <= 400: return 'High (301-400)'
    else:          return 'Very High (400+)'

def hazard_lvl(r):
    if   r <= 5:  return 'Low Hazard'
    elif r <= 15: return 'Moderate Hazard'
    elif r <= 30: return 'High Hazard'
    else:         return 'Critical Hazard'

def veh_class(v):
    return {'Bicycle':'Non-Motorised','Motorcycle':'Two-Wheeler',
            'Car':'Light Motor','Bus':'Heavy Motor'
            ,'Truck':'Heavy Motor'}.get(v,'Other')

print(' Helper functions defined.')

 Helper functions defined.


In [38]:
# ─────────────────────────────────────────────────────────────────────────────
# TRANSFORM — Sensor Data
# ─────────────────────────────────────────────────────────────────────────────
sensor = raw_sensor.copy()
sensor['Date_Time'] = pd.to_datetime(sensor['Date_Time'])

# Datetime decomposition for dim_date
sensor['date_only']        = sensor['Date_Time'].dt.date
sensor['year']             = sensor['Date_Time'].dt.year
sensor['quarter']          = sensor['Date_Time'].dt.quarter
sensor['month']            = sensor['Date_Time'].dt.month
sensor['month_name']       = sensor['Date_Time'].dt.strftime('%B')
sensor['month_sort']       = sensor['Date_Time'].dt.month            # numeric sort key
sensor['week']             = sensor['Date_Time'].dt.isocalendar().week.astype(int)
sensor['day_name']         = sensor['Date_Time'].dt.day_name()
sensor['day_of_week_num']  = sensor['Date_Time'].dt.dayofweek + 1   # Mon=1 … Sun=7
sensor['hour']             = sensor['Date_Time'].dt.hour
sensor['time_of_day']      = sensor['hour'].apply(time_of_day)
sensor['is_peak_hour']     = sensor['time_of_day'].isin(['Morning Peak','Evening Peak']).astype(int)
sensor['is_weekend']       = sensor['day_name'].isin(['Saturday','Sunday']).astype(int)

# ── KPI: Congestion Index ─────────────────────────────────────────────────────
# Formula: 0.6 × (vehicle_count / max_vehicle_count)
#        + 0.4 × (1 − average_speed / max_speed)
# Range: 0 (free flow) → 1 (standstill/gridlock)
# Weights: volume contributes 60%, inverse-speed 40%
# Anchored to dataset max values so all readings are comparable.
MAX_VOL = raw_sensor['Vehicle_Count'].max()   # 494
MAX_SPD = raw_sensor['Average_Speed'].max()   # 79

sensor['congestion_index']    = (
    0.6 * (sensor['Vehicle_Count'] / MAX_VOL) +
    0.4 * (1 - sensor['Average_Speed'] / MAX_SPD)
).round(4)
sensor['congestion_level_num']= sensor['Congestion_Level'].map({'Low':1,'Moderate':2,'High':3})
sensor['speed_category']      = sensor['Average_Speed'].apply(speed_cat)
sensor['volume_band']         = sensor['Vehicle_Count'].apply(volume_band)
sensor['sensor_status']       = 'Active'   # Assumption: all sensors are Active

print(' Sensor transform complete.')
print(f'   Congestion Index range: {sensor["congestion_index"].min():.4f} – {sensor["congestion_index"].max():.4f}')
display(sensor[['Sensor_ID','Location','Vehicle_Count','Average_Speed',
                'Congestion_Level','congestion_index','speed_category','time_of_day']].head(4))

 Sensor transform complete.
   Congestion Index range: 0.0976 – 0.8813


,Sensor_ID,Location,Vehicle_Count,Average_Speed,Congestion_Level,congestion_index,speed_category,time_of_day
0,a36f14,Downtown,90,59,High,0.211,Moderate (40-59 kmh),Off-Peak Night
1,256a39,Downtown,324,73,Moderate,0.424,Fast (60-79 kmh),Off-Peak Night
2,a8a01f,Highway,61,56,Moderate,0.191,Moderate (40-59 kmh),Off-Peak Night
3,3036e3,Highway,90,56,High,0.226,Moderate (40-59 kmh),Off-Peak Night


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TRANSFORM — Accident Data
# ─────────────────────────────────────────────────────────────────────────────
accident = raw_accident.copy()
accident['Date_Time'] = pd.to_datetime(accident['Date_Time'])

# Datetime decomposition for dim_date
accident['date_only']        = accident['Date_Time'].dt.date
accident['year']             = accident['Date_Time'].dt.year
accident['quarter']          = accident['Date_Time'].dt.quarter
accident['month']            = accident['Date_Time'].dt.month
accident['month_name']       = accident['Date_Time'].dt.strftime('%B')
accident['month_sort']       = accident['Date_Time'].dt.month
accident['week']             = accident['Date_Time'].dt.isocalendar().week.astype(int)
accident['day_name']         = accident['Date_Time'].dt.day_name()
accident['day_of_week_num']  = accident['Date_Time'].dt.dayofweek + 1
accident['hour']             = accident['Date_Time'].dt.hour
accident['time_of_day']      = accident['hour'].apply(time_of_day)
accident['is_peak_hour']     = accident['time_of_day'].isin(['Morning Peak','Evening Peak']).astype(int)
accident['is_weekend']       = accident['day_name'].isin(['Saturday','Sunday']).astype(int)

#  KPI: Severity Score 
SEV_MAP = {'Minor':1, 'Moderate':2, 'Severe':3, 'Fatal':4}
accident['severity_score'] = accident['Accident_Severity'].map(SEV_MAP)

# KPI: Risk Score
# Formula: severity_score × log1p(casualties + 1) × number_of_vehicles
# WHY log1p(casualties + 1) and not log1p(casualties):
#   501 accidents have Casualties = 0, including 120 Fatal accidents.
#   log1p(0) = 0, so without +1 any zero-casualty Fatal accident would get
#   risk_score = 0 — indistinguishable from a zero-casualty Minor accident.
#   Adding +1 gives log1p(1) = 0.693 as the floor, preserving severity signal.
accident['risk_score'] = (
    accident['severity_score'] *
    np.log1p(accident['Casualties'] + 1) *
    accident['Number_of_Vehicles']
).round(4)

# Derived flag columns 
accident['hazard_level']       = accident['risk_score'].apply(hazard_lvl)
accident['adverse_weather']    = accident['Weather_Condition'].isin(['Fog','Snow','Storm']).astype(int)
accident['hazardous_road']     = accident['Road_Condition'].isin(['Icy','Under Construction']).astype(int)
accident['is_fatal_or_severe'] = accident['Accident_Severity'].isin(['Fatal','Severe']).astype(int)
accident['traffic_density_num']= accident['Traffic_Density'].map({'Low':1,'Moderate':2,'High':3})

print(' Accident transform complete.')
print(f'   Risk Score range  : {accident["risk_score"].min():.3f} – {accident["risk_score"].max():.3f}')
print(f'   Fatal/Severe rows : {accident["is_fatal_or_severe"].sum():,} ({accident["is_fatal_or_severe"].mean()*100:.1f}%)')
print(f'   Hazard distribution:')
print(accident['hazard_level'].value_counts().to_string())
display(accident[['Accident_ID','Location','Accident_Severity','Casualties',
                'Number_of_Vehicles','severity_score','risk_score','hazard_level']].head(4))

 Accident transform complete.
   Risk Score range  : 0.693 – 38.366
   Fatal/Severe rows : 2,490 (49.8%)
   Hazard distribution:
hazard_level
Moderate Hazard    2192
Low Hazard         1466
High Hazard        1192
Critical Hazard     150


,Accident_ID,Location,Accident_Severity,Casualties,Number_of_Vehicles,severity_score,risk_score,hazard_level
0,d02b3836,Suburbs,Minor,6,2,1,4.159,Low Hazard
1,694c8577,Downtown,Minor,2,2,1,2.773,Low Hazard
2,e0ed5c55,Residential Zone,Moderate,5,4,2,15.567,High Hazard
3,6a78511a,Downtown,Moderate,8,3,2,13.816,Moderate Hazard


---
## 📤 Section 5 — ETL: Load (Write Directly to MySQL)

In [40]:
# ── LOAD: dim_location ───────────────────────────────────────────────────────
ZONE_MAP = {
    'Downtown'        : 'Urban Core',
    'Highway'         : 'Highway',
    'Residential Zone': 'Residential',
    'Industrial Area' : 'Industrial',
    'Suburbs'         : 'Suburban'
}
locs     = sorted(set(sensor['Location'].unique()) | set(accident['Location'].unique()))
loc_rows = [(loc, ZONE_MAP.get(loc, 'Other')) for loc in locs]

conn = get_conn(); cur = conn.cursor()
exec_sql(cur, 'INSERT IGNORE INTO dim_location (location_name, zone_type) VALUES (%s,%s)',
        loc_rows, many=True)
conn.commit()
cur.execute('SELECT location_name, location_id FROM dim_location')
LOC_ID = {r[0]: r[1] for r in cur.fetchall()}
conn.close()
print(f' dim_location → {len(loc_rows)} rows')
display(pd.DataFrame(loc_rows, columns=['Location', 'Zone_Type']))

 dim_location → 5 rows


,Location,Zone_Type
0,Downtown,Urban Core
1,Highway,Highway
2,Industrial Area,Industrial
3,Residential Zone,Residential
4,Suburbs,Suburban


In [41]:
# ── LOAD: dim_vehicle ────────────────────────────────────────────────────────
veh_rows = [(v, veh_class(v)) for v in sorted(accident['Vehicle_Type'].unique())]

conn = get_conn(); cur = conn.cursor()
exec_sql(cur, 'INSERT IGNORE INTO dim_vehicle (vehicle_type, vehicle_class) VALUES (%s,%s)',
        veh_rows, many=True)
conn.commit()
cur.execute('SELECT vehicle_type, vehicle_id FROM dim_vehicle')
VEH_ID = {r[0]: r[1] for r in cur.fetchall()}
conn.close()
print(f' dim_vehicle → {len(veh_rows)} rows')
display(pd.DataFrame(veh_rows, columns=['Vehicle_Type', 'Vehicle_Class']))

 dim_vehicle → 5 rows


,Vehicle_Type,Vehicle_Class
0,Bicycle,Non-Motorised
1,Bus,Heavy Motor
2,Car,Light Motor
3,Motorcycle,Two-Wheeler
4,Truck,Heavy Motor


In [42]:
# ── LOAD: dim_weather ────────────────────────────────────────────────────────
ADVERSE   = {'Fog', 'Snow', 'Storm'}
wth_rows  = [(w, 1 if w in ADVERSE else 0)
            for w in sorted(accident['Weather_Condition'].unique())]

conn = get_conn(); cur = conn.cursor()
exec_sql(cur, 'INSERT IGNORE INTO dim_weather (weather_condition, is_adverse) VALUES (%s,%s)',
        wth_rows, many=True)
conn.commit()
cur.execute('SELECT weather_condition, weather_id FROM dim_weather')
WTH_ID = {r[0]: r[1] for r in cur.fetchall()}
conn.close()
print(f' dim_weather → {len(wth_rows)} rows')
display(pd.DataFrame(wth_rows, columns=['Weather_Condition', 'Is_Adverse']))

 dim_weather → 5 rows


,Weather_Condition,Is_Adverse
0,Clear,0
1,Fog,1
2,Rain,0
3,Snow,1
4,Storm,1


In [43]:
# ── LOAD: dim_date ────────────────────────────────────────────────────────────
# Union all unique timestamps from both datasets.
# All 300 sensor timestamps are a subset of the 5,000 accident timestamps,
# so the combined set has exactly 5,000 unique rows.
# FIX: date_only is passed as a proper datetime.date object, NOT a string,
#      so MySQL stores it as DATE type correctly.
DATE_COLS = ['Date_Time','date_only','year','quarter','month','month_name','month_sort',
             'week','day_name','day_of_week_num','hour','time_of_day','is_peak_hour','is_weekend']

dim_date_df = (
    pd.concat([sensor[DATE_COLS], accident[DATE_COLS]])
    .drop_duplicates(subset='Date_Time')
    .reset_index(drop=True)
)

date_rows = [
    (str(r['Date_Time']),          # full_datetime  → stored as DATETIME
     r['date_only'],               # date_only       → passed as date object → DATE type
     int(r['year']), int(r['quarter']), int(r['month']),
     r['month_name'], int(r['month_sort']),
     int(r['week']), r['day_name'], int(r['day_of_week_num']),
     int(r['hour']), r['time_of_day'], int(r['is_peak_hour']), int(r['is_weekend']))
    for _, r in dim_date_df.iterrows()
]

conn = get_conn(); cur = conn.cursor()
exec_sql(cur,
    '''INSERT IGNORE INTO dim_date
       (full_datetime, date_only, year, quarter, month, month_name, month_sort,
        week, day_name, day_of_week_num, hour, time_of_day, is_peak_hour, is_weekend)
       VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)''',
    date_rows, many=True)
conn.commit()
cur.execute('SELECT full_datetime, date_id FROM dim_date')
DATE_ID = {str(r[0]): r[1] for r in cur.fetchall()}
conn.close()
print(f' dim_date → {len(date_rows):,} rows')

 dim_date → 5,000 rows


In [44]:
# ── LOAD: fact_traffic ────────────────────────────────────────────────────────
sensor['location_id'] = sensor['Location'].map(LOC_ID)
sensor['date_id']     = sensor['Date_Time'].astype(str).map(DATE_ID)

traffic_rows = [
    (r['Sensor_ID'], int(r['location_id']), int(r['date_id']),
     int(r['Vehicle_Count']), int(r['Average_Speed']),
     r['Congestion_Level'], int(r['congestion_level_num']),
     float(r['congestion_index']), r['speed_category'], r['volume_band'], r['sensor_status'])
    for _, r in sensor.iterrows()
]
conn = get_conn(); cur = conn.cursor()
exec_sql(cur,
    '''INSERT INTO fact_traffic
       (sensor_id,location_id,date_id,vehicle_count,average_speed,
        congestion_level,congestion_level_num,congestion_index,
        speed_category,volume_band,sensor_status)
       VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)''',
    traffic_rows, many=True)
conn.commit(); conn.close()
print(f' fact_traffic → {len(traffic_rows):,} rows')

 fact_traffic → 300 rows


In [45]:
# ── LOAD: fact_accident ───────────────────────────────────────────────────────
# NOTE: vehicle_type and weather_condition are inserted BOTH via FK (vehicle_id,
# weather_id) AND as denormalised VARCHAR columns. This lets Power BI slicers
# filter directly on fact_accident without requiring a JOIN to dim tables.
accident['location_id'] = accident['Location'].map(LOC_ID)
accident['date_id']     = accident['Date_Time'].astype(str).map(DATE_ID)
accident['vehicle_id']  = accident['Vehicle_Type'].map(VEH_ID)
accident['weather_id']  = accident['Weather_Condition'].map(WTH_ID)

accident_rows = [
    (r['Accident_ID'],
     int(r['location_id']), int(r['date_id']),
     int(r['vehicle_id']), int(r['weather_id']),
     r['Vehicle_Type'], r['Weather_Condition'], r['Road_Condition'],
     r['Accident_Severity'], int(r['Number_of_Vehicles']), int(r['Casualties']),
     r['Traffic_Density'],
     int(r['severity_score']), float(r['risk_score']), r['hazard_level'],
     int(r['adverse_weather']), int(r['hazardous_road']),
     int(r['is_peak_hour']), int(r['is_fatal_or_severe']), int(r['traffic_density_num']))
    for _, r in accident.iterrows()
]
conn = get_conn(); cur = conn.cursor()
exec_sql(cur,
    '''INSERT INTO fact_accident
       (accident_id,location_id,date_id,vehicle_id,weather_id,
        vehicle_type,weather_condition,road_condition,
        accident_severity,number_of_vehicles,casualties,traffic_density,
        severity_score,risk_score,hazard_level,
        adverse_weather,hazardous_road,is_peak_hour,
        is_fatal_or_severe,traffic_density_num)
       VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)''',
    accident_rows, many=True)
conn.commit(); conn.close()
print(f' fact_accident → {len(accident_rows):,} rows')

print('\n ETL LOAD COMPLETE')
for t,n in [('dim_location', len(locs)), ('dim_date', len(date_rows)),
            ('dim_vehicle', len(veh_rows)), ('dim_weather', len(wth_rows)),
            ('fact_traffic', len(traffic_rows)), ('fact_accident', len(accident_rows))]:
    print(f'   {t:<25} {n:>7,} rows')

 fact_accident → 5,000 rows

 ETL LOAD COMPLETE
   dim_location                    5 rows
   dim_date                    5,000 rows
   dim_vehicle                     5 rows
   dim_weather                     5 rows
   fact_traffic                  300 rows
   fact_accident               5,000 rows


---
## 🔍 Section 6 — SQL Analysis Queries

In [46]:
# Q1 — Traffic Volume & Speed by Location
qdf("""
    SELECT  l.location_name                       AS Location,
            l.zone_type                           AS Zone,
            COUNT(*)                              AS Total_Readings,
            COUNT(DISTINCT t.sensor_id)           AS Active_Sensors,
            ROUND(AVG(t.vehicle_count),1)         AS Avg_Vehicles,
            MAX(t.vehicle_count)                  AS Peak_Vehicles,
            ROUND(AVG(t.average_speed),1)         AS Avg_Speed_kmh,
            ROUND(MIN(t.average_speed),1)         AS Min_Speed_kmh,
            ROUND(AVG(t.congestion_index),4)      AS Avg_Congestion_Index,
            ROUND(MAX(t.congestion_index),4)      AS Max_Congestion_Index
    FROM    fact_traffic t
    JOIN    dim_location l ON t.location_id = l.location_id
    GROUP BY l.location_name, l.zone_type
    ORDER BY Avg_Congestion_Index DESC
""", 'Q1 — Traffic Volume & Speed by Location')


── Q1 — Traffic Volume & Speed by Location ──


,Location,Zone,Total_Readings,Active_Sensors,Avg_Vehicles,Peak_Vehicles,Avg_Speed_kmh,Min_Speed_kmh,Avg_Congestion_Index,Max_Congestion_Index
0,Residential Zone,Residential,59,59,329.400,494,45.500,21,0.570,0.881
1,Downtown,Urban Core,55,55,290.600,492,49.400,20,0.503,0.854
2,Industrial Area,Industrial,51,51,283.800,494,50.200,20,0.491,0.858
3,Highway,Highway,80,80,257.800,491,49.700,21,0.461,0.813
4,Suburbs,Suburban,55,55,238.400,477,50.800,20,0.432,0.873


,Location,Zone,Total_Readings,Active_Sensors,Avg_Vehicles,Peak_Vehicles,Avg_Speed_kmh,Min_Speed_kmh,Avg_Congestion_Index,Max_Congestion_Index
0,Residential Zone,Residential,59,59,329.400,494,45.500,21,0.570,0.881
1,Downtown,Urban Core,55,55,290.600,492,49.400,20,0.503,0.854
2,Industrial Area,Industrial,51,51,283.800,494,50.200,20,0.491,0.858
3,Highway,Highway,80,80,257.800,491,49.700,21,0.461,0.813
4,Suburbs,Suburban,55,55,238.400,477,50.800,20,0.432,0.873


In [47]:
# Q2 — Congestion Level Distribution by Location
qdf("""
    SELECT  l.location_name                                                AS Location,
            t.congestion_level                                             AS Level,
            COUNT(*)                                                       AS Count,
            ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY l.location_name), 1) AS Pct
    FROM    fact_traffic t
    JOIN    dim_location l ON t.location_id = l.location_id
    GROUP BY l.location_name, t.congestion_level, t.congestion_level_num
    ORDER BY l.location_name, t.congestion_level_num DESC
""", 'Q2 — Congestion Level Distribution by Location')


── Q2 — Congestion Level Distribution by Location ──


,Location,Level,Count,Pct
0,Downtown,High,18,32.700
1,Downtown,Moderate,23,41.800
2,Downtown,Low,14,25.500
3,Highway,High,26,32.500
4,Highway,Moderate,26,32.500
5,Highway,Low,28,35.000
6,Industrial Area,High,13,25.500
7,Industrial Area,Moderate,15,29.400
8,Industrial Area,Low,23,45.100
9,Residential Zone,High,14,23.700


,Location,Level,Count,Pct
0,Downtown,High,18,32.700
1,Downtown,Moderate,23,41.800
2,Downtown,Low,14,25.500
3,Highway,High,26,32.500
4,Highway,Moderate,26,32.500
5,Highway,Low,28,35.000
6,Industrial Area,High,13,25.500
7,Industrial Area,Moderate,15,29.400
8,Industrial Area,Low,23,45.100
9,Residential Zone,High,14,23.700


In [ ]:
# Q3 — 24-Hour Traffic Profile (Peak Hour Identification)
qdf("""
    SELECT  d.hour,
            d.time_of_day,
            d.is_peak_hour,
            ROUND(AVG(t.vehicle_count),1)     AS Avg_Vehicles,
            ROUND(AVG(t.average_speed),1)     AS Avg_Speed,
            ROUND(AVG(t.congestion_index),4)  AS Avg_CI,
            SUM(t.congestion_level = 'High')  AS High_Cong_Count
    FROM    fact_traffic t
    JOIN    dim_date     d ON t.date_id = d.date_id
    GROUP BY d.hour, d.time_of_day, d.is_peak_hour
    ORDER BY d.hour
""", 'Q3 — 24-Hour Traffic Profile')


── Q3 — 24-Hour Traffic Profile ──


,hour,time_of_day,is_peak_hour,Avg_Vehicles,Avg_Speed,Avg_CI,High_Cong_Count
0,0,Off-Peak Night,0,243.000,49.500,0.444,3.000
1,1,Off-Peak Night,0,286.200,50.700,0.491,6.000
2,2,Off-Peak Night,0,312.500,48.400,0.535,4.000
3,3,Off-Peak Night,0,269.400,41.300,0.518,4.000
4,4,Off-Peak Night,0,305.700,53.600,0.500,5.000
5,5,Off-Peak Night,0,280.100,50.800,0.483,5.000
6,6,Off-Peak Day,0,277.200,48.300,0.492,4.000
7,7,Morning Peak,1,281.600,53.000,0.474,3.000
8,8,Morning Peak,1,256.900,40.100,0.509,2.000
9,9,Morning Peak,1,255.200,50.800,0.453,5.000


,hour,time_of_day,is_peak_hour,Avg_Vehicles,Avg_Speed,Avg_CI,High_Cong_Count
0,0,Off-Peak Night,0,243.000,49.500,0.444,3.000
1,1,Off-Peak Night,0,286.200,50.700,0.491,6.000
2,2,Off-Peak Night,0,312.500,48.400,0.535,4.000
3,3,Off-Peak Night,0,269.400,41.300,0.518,4.000
4,4,Off-Peak Night,0,305.700,53.600,0.500,5.000
5,5,Off-Peak Night,0,280.100,50.800,0.483,5.000
6,6,Off-Peak Day,0,277.200,48.300,0.492,4.000
7,7,Morning Peak,1,281.600,53.000,0.474,3.000
8,8,Morning Peak,1,256.900,40.100,0.509,2.000
9,9,Morning Peak,1,255.200,50.800,0.453,5.000


In [49]:
# Q4 — Peak Hour Congestion Hotspots by Location
qdf("""
    SELECT  l.location_name,
            l.zone_type,
            d.time_of_day,
            COUNT(*)                                                        AS Readings,
            ROUND(AVG(t.vehicle_count),1)                                   AS Avg_Vehicles,
            ROUND(AVG(t.congestion_index),4)                                AS Avg_CI,
            SUM(t.congestion_level = 'High')                                AS High_Hits,
            ROUND(100.0 * SUM(t.congestion_level='High') / COUNT(*), 1)    AS High_Pct
    FROM    fact_traffic t
    JOIN    dim_location l ON t.location_id = l.location_id
    JOIN    dim_date     d ON t.date_id     = d.date_id
    WHERE   d.is_peak_hour = 1
    GROUP BY l.location_name, l.zone_type, d.time_of_day
    ORDER BY Avg_CI DESC
""", 'Q4 — Peak Hour Congestion Hotspots')


── Q4 — Peak Hour Congestion Hotspots ──


,location_name,zone_type,time_of_day,Readings,Avg_Vehicles,Avg_CI,High_Hits,High_Pct
0,Residential Zone,Residential,Morning Peak,9,360.100,0.593,1.000,11.100
1,Residential Zone,Residential,Evening Peak,13,321.500,0.579,2.000,15.400
2,Industrial Area,Industrial,Evening Peak,8,338.300,0.579,1.000,12.500
3,Industrial Area,Industrial,Morning Peak,10,302.600,0.535,3.000,30.000
4,Highway,Highway,Evening Peak,8,313.000,0.509,4.000,50.000
5,Suburbs,Suburban,Morning Peak,10,254.900,0.486,4.000,40.000
6,Downtown,Urban Core,Morning Peak,11,255.500,0.485,2.000,18.200
7,Downtown,Urban Core,Evening Peak,10,275.800,0.436,4.000,40.000
8,Highway,Highway,Morning Peak,12,233.600,0.434,4.000,33.300
9,Suburbs,Suburban,Evening Peak,9,208.200,0.341,3.000,33.300


,location_name,zone_type,time_of_day,Readings,Avg_Vehicles,Avg_CI,High_Hits,High_Pct
0,Residential Zone,Residential,Morning Peak,9,360.100,0.593,1.000,11.100
1,Residential Zone,Residential,Evening Peak,13,321.500,0.579,2.000,15.400
2,Industrial Area,Industrial,Evening Peak,8,338.300,0.579,1.000,12.500
3,Industrial Area,Industrial,Morning Peak,10,302.600,0.535,3.000,30.000
4,Highway,Highway,Evening Peak,8,313.000,0.509,4.000,50.000
5,Suburbs,Suburban,Morning Peak,10,254.900,0.486,4.000,40.000
6,Downtown,Urban Core,Morning Peak,11,255.500,0.485,2.000,18.200
7,Downtown,Urban Core,Evening Peak,10,275.800,0.436,4.000,40.000
8,Highway,Highway,Morning Peak,12,233.600,0.434,4.000,33.300
9,Suburbs,Suburban,Evening Peak,9,208.200,0.341,3.000,33.300


In [ ]:
# Q5 — Accident-Prone Areas Ranked by Total Risk Score
qdf("""
    SELECT  l.location_name,
            l.zone_type,
            COUNT(*)                                               AS Total_Accidents,
            SUM(a.casualties)                                      AS Total_Casualties,
            ROUND(AVG(a.risk_score),3)                             AS Avg_Risk_Score,
            ROUND(SUM(a.risk_score),2)                             AS Total_Risk_Score,
            SUM(a.accident_severity = 'Fatal')                     AS Fatal,
            SUM(a.accident_severity = 'Severe')                    AS Severe,
            ROUND(100.0 * SUM(a.is_fatal_or_severe) / COUNT(*), 1) AS Fatal_Severe_Pct
    FROM    fact_accident a
    JOIN    dim_location  l ON a.location_id = l.location_id
    GROUP BY l.location_name, l.zone_type
    ORDER BY Total_Risk_Score DESC
""", 'Q5 — Accident-Prone Areas by Total Risk Score')


── Q5 — Accident-Prone Areas by Total Risk Score ──


,location_name,zone_type,Total_Accidents,Total_Casualties,Avg_Risk_Score,Total_Risk_Score,Fatal,Severe,Fatal_Severe_Pct
0,Highway,Highway,1018,4666.000,11.404,11608.780,260.000,244.000,49.500
1,Residential Zone,Residential,998,4451.000,11.205,11182.920,243.000,260.000,50.400
2,Industrial Area,Industrial,993,4465.000,11.234,11155.760,253.000,275.000,53.200
3,Suburbs,Suburban,1017,4480.000,10.569,10748.830,275.000,225.000,49.200
4,Downtown,Urban Core,974,4405.000,10.777,10496.660,222.000,233.000,46.700


,location_name,zone_type,Total_Accidents,Total_Casualties,Avg_Risk_Score,Total_Risk_Score,Fatal,Severe,Fatal_Severe_Pct
0,Highway,Highway,1018,4666.000,11.404,11608.780,260.000,244.000,49.500
1,Residential Zone,Residential,998,4451.000,11.205,11182.920,243.000,260.000,50.400
2,Industrial Area,Industrial,993,4465.000,11.234,11155.760,253.000,275.000,53.200
3,Suburbs,Suburban,1017,4480.000,10.569,10748.830,275.000,225.000,49.200
4,Downtown,Urban Core,974,4405.000,10.777,10496.660,222.000,233.000,46.700


In [ ]:
# Q6 — Weather × Road Condition Risk Matrix
qdf("""
    SELECT  w.weather_condition,
            w.is_adverse                                      AS Is_Adverse_Weather,
            a.road_condition,
            a.hazardous_road                                  AS Is_Hazardous_Road,
            COUNT(*)                                          AS Accidents,
            SUM(a.casualties)                                 AS Casualties,
            ROUND(AVG(a.severity_score),2)                    AS Avg_Severity,
            ROUND(AVG(a.risk_score),3)                        AS Avg_Risk,
            SUM(a.accident_severity = 'Fatal')                AS Fatal_Count,
            ROUND(100.0*SUM(a.is_fatal_or_severe)/COUNT(*),1) AS Fatal_Severe_Pct
    FROM    fact_accident a
    JOIN    dim_weather   w ON a.weather_id = w.weather_id
    GROUP BY w.weather_condition, w.is_adverse, a.road_condition, a.hazardous_road
    ORDER BY Avg_Risk DESC
""", 'Q6 — Weather & Road Condition Risk Matrix')


── Q6 — Weather & Road Condition Risk Matrix ──


,weather_condition,Is_Adverse_Weather,road_condition,Is_Hazardous_Road,Accidents,Casualties,Avg_Severity,Avg_Risk,Fatal_Count,Fatal_Severe_Pct
0,Clear,0,Dry,0,242,1129.000,2.620,12.216,76.000,55.800
1,Rain,0,Dry,0,269,1187.000,2.570,11.719,67.000,53.900
2,Storm,1,Under Construction,1,220,1022.000,2.450,11.654,55.000,46.400
3,Clear,0,Icy,1,277,1260.000,2.560,11.502,71.000,53.800
4,Snow,1,Dry,0,253,1090.000,2.590,11.288,65.000,54.500
5,Clear,0,Wet,0,241,1029.000,2.460,11.211,61.000,47.700
6,Storm,1,Icy,1,240,1089.000,2.430,11.210,60.000,46.700
7,Rain,0,Under Construction,1,241,1108.000,2.510,11.210,57.000,49.400
8,Snow,1,Wet,0,254,1153.000,2.530,11.167,66.000,52.800
9,Rain,0,Icy,1,276,1221.000,2.440,11.035,62.000,47.100


,weather_condition,Is_Adverse_Weather,road_condition,Is_Hazardous_Road,Accidents,Casualties,Avg_Severity,Avg_Risk,Fatal_Count,Fatal_Severe_Pct
0,Clear,0,Dry,0,242,1129.000,2.620,12.216,76.000,55.800
1,Rain,0,Dry,0,269,1187.000,2.570,11.719,67.000,53.900
2,Storm,1,Under Construction,1,220,1022.000,2.450,11.654,55.000,46.400
3,Clear,0,Icy,1,277,1260.000,2.560,11.502,71.000,53.800
4,Snow,1,Dry,0,253,1090.000,2.590,11.288,65.000,54.500
5,Clear,0,Wet,0,241,1029.000,2.460,11.211,61.000,47.700
6,Storm,1,Icy,1,240,1089.000,2.430,11.210,60.000,46.700
7,Rain,0,Under Construction,1,241,1108.000,2.510,11.210,57.000,49.400
8,Snow,1,Wet,0,254,1153.000,2.530,11.167,66.000,52.800
9,Rain,0,Icy,1,276,1221.000,2.440,11.035,62.000,47.100


In [ ]:
# Q7 — Monthly Accident Trend
qdf("""
    SELECT  d.year, d.quarter, d.month, d.month_name,
            COUNT(*)                             AS Accidents,
            SUM(a.casualties)                    AS Casualties,
            ROUND(AVG(a.risk_score),3)           AS Avg_Risk,
            SUM(a.accident_severity = 'Fatal')   AS Fatal,
            SUM(a.accident_severity = 'Severe')  AS Severe,
            SUM(a.adverse_weather)               AS Adverse_Weather_Accidents
    FROM    fact_accident a
    JOIN    dim_date      d ON a.date_id = d.date_id
    GROUP BY d.year, d.quarter, d.month, d.month_name
    ORDER BY d.year, d.month
""", 'Q7 — Monthly Accident Trend')


── Q7 — Monthly Accident Trend ──


,year,quarter,month,month_name,Accidents,Casualties,Avg_Risk,Fatal,Severe,Adverse_Weather_Accidents
0,2024,1,1,January,744,3417.000,11.238,202.000,183.000,434.000
1,2024,1,2,February,696,3129.000,10.786,181.000,165.000,426.000
2,2024,1,3,March,744,3244.000,10.948,171.000,190.000,416.000
3,2024,2,4,April,720,3353.000,11.407,195.000,179.000,420.000
4,2024,2,5,May,744,3314.000,10.470,163.000,184.000,429.000
5,2024,2,6,June,720,3221.000,10.974,173.000,162.000,434.000
6,2024,3,7,July,632,2789.000,11.512,168.000,174.000,368.000


,year,quarter,month,month_name,Accidents,Casualties,Avg_Risk,Fatal,Severe,Adverse_Weather_Accidents
0,2024,1,1,January,744,3417.000,11.238,202.000,183.000,434.000
1,2024,1,2,February,696,3129.000,10.786,181.000,165.000,426.000
2,2024,1,3,March,744,3244.000,10.948,171.000,190.000,416.000
3,2024,2,4,April,720,3353.000,11.407,195.000,179.000,420.000
4,2024,2,5,May,744,3314.000,10.470,163.000,184.000,429.000
5,2024,2,6,June,720,3221.000,10.974,173.000,162.000,434.000
6,2024,3,7,July,632,2789.000,11.512,168.000,174.000,368.000


In [53]:
# Q8 — Vehicle Type Risk Profile (via dim_vehicle)
qdf("""
    SELECT  v.vehicle_type,
            v.vehicle_class,
            COUNT(*)                                        AS Accidents,
            SUM(a.casualties)                               AS Total_Casualties,
            ROUND(AVG(a.number_of_vehicles),1)              AS Avg_Vehicles_Involved,
            ROUND(AVG(a.severity_score),2)                  AS Avg_Severity,
            ROUND(AVG(a.risk_score),3)                      AS Avg_Risk_Score,
            SUM(a.accident_severity = 'Fatal')              AS Fatal_Count
    FROM    fact_accident a
    JOIN    dim_vehicle   v ON a.vehicle_id = v.vehicle_id
    GROUP BY v.vehicle_type, v.vehicle_class
    ORDER BY Avg_Risk_Score DESC
""", 'Q8 — Vehicle Type Risk Profile')


── Q8 — Vehicle Type Risk Profile ──


,vehicle_type,vehicle_class,Accidents,Total_Casualties,Avg_Vehicles_Involved,Avg_Severity,Avg_Risk_Score,Fatal_Count
0,Bus,Heavy Motor,956,4228.000,2.500,2.500,11.345,247.000
1,Motorcycle,Two-Wheeler,972,4465.000,2.500,2.500,11.202,239.000
2,Car,Light Motor,1032,4621.000,2.500,2.540,11.082,273.000
3,Bicycle,Non-Motorised,988,4465.000,2.500,2.540,11.071,256.000
4,Truck,Heavy Motor,1052,4688.000,2.500,2.400,10.535,238.000


,vehicle_type,vehicle_class,Accidents,Total_Casualties,Avg_Vehicles_Involved,Avg_Severity,Avg_Risk_Score,Fatal_Count
0,Bus,Heavy Motor,956,4228.000,2.500,2.500,11.345,247.000
1,Motorcycle,Two-Wheeler,972,4465.000,2.500,2.500,11.202,239.000
2,Car,Light Motor,1032,4621.000,2.500,2.540,11.082,273.000
3,Bicycle,Non-Motorised,988,4465.000,2.500,2.540,11.071,256.000
4,Truck,Heavy Motor,1052,4688.000,2.500,2.400,10.535,238.000


In [54]:
# Q9 — Day-of-Week Traffic & Accident Pattern (sorted Mon–Sun via day_of_week_num)
qdf("""
    SELECT  d.day_name,
            d.day_of_week_num,
            d.is_weekend,
            COUNT(DISTINCT t.traffic_id)                    AS Sensor_Readings,
            ROUND(AVG(t.vehicle_count),1)                   AS Avg_Vehicles,
            ROUND(AVG(t.congestion_index),4)                AS Avg_CI,
            COUNT(DISTINCT a.accident_pk)                   AS Accidents,
            COALESCE(SUM(a.casualties),0)                   AS Casualties
    FROM         dim_date     d
    LEFT JOIN    fact_traffic  t ON d.date_id = t.date_id
    LEFT JOIN    fact_accident a ON d.date_id = a.date_id
    GROUP BY d.day_name, d.day_of_week_num, d.is_weekend
    ORDER BY d.day_of_week_num
""", 'Q9 — Day-of-Week Traffic & Accident Pattern (Mon→Sun)')


── Q9 — Day-of-Week Traffic & Accident Pattern (Mon→Sun) ──


,day_name,day_of_week_num,is_weekend,Sensor_Readings,Avg_Vehicles,Avg_CI,Accidents,Casualties
0,Monday,1,0,48,279.900,0.494,720,3232.000
1,Tuesday,2,0,48,267.100,0.467,720,3154.000
2,Wednesday,3,0,48,290.300,0.520,720,3259.000
3,Thursday,4,0,48,301.900,0.502,720,3246.000
4,Friday,5,0,48,294.800,0.517,720,3254.000
5,Saturday,6,1,36,255.000,0.446,704,3122.000
6,Sunday,7,1,24,233.900,0.455,696,3200.000


,day_name,day_of_week_num,is_weekend,Sensor_Readings,Avg_Vehicles,Avg_CI,Accidents,Casualties
0,Monday,1,0,48,279.900,0.494,720,3232.000
1,Tuesday,2,0,48,267.100,0.467,720,3154.000
2,Wednesday,3,0,48,290.300,0.520,720,3259.000
3,Thursday,4,0,48,301.900,0.502,720,3246.000
4,Friday,5,0,48,294.800,0.517,720,3254.000
5,Saturday,6,1,36,255.000,0.446,704,3122.000
6,Sunday,7,1,24,233.900,0.455,696,3200.000


---
## 📊 Section 7 — KPI Summary (Computed from MySQL)

In [63]:
print('=' * 52)
print('KPI DASHBOARD — MetroCity Smart Traffic System')
print('=' * 52)

qdf("""
    SELECT  COUNT(*)                                                  AS Total_Sensor_Readings,
            COUNT(DISTINCT sensor_id)                                 AS Active_Sensors,
            SUM(vehicle_count)                                        AS Total_Vehicles_Recorded,
            ROUND(AVG(congestion_index),4)                            AS Avg_Congestion_Index,
            ROUND(100.0 * SUM(congestion_level='High')/COUNT(*),2)   AS High_Congestion_Pct,
            ROUND(AVG(average_speed),1)                               AS Avg_Speed_kmh,
            MAX(vehicle_count)                                        AS Peak_Vehicle_Count
    FROM    fact_traffic
""", 'Traffic KPIs')

qdf("""
    SELECT  COUNT(*)                                                       AS Total_Accidents,
            SUM(casualties)                                                AS Total_Casualties,
            ROUND(AVG(risk_score),3)                                       AS Avg_Risk_Score,
            ROUND(MAX(risk_score),3)                                       AS Max_Risk_Score,
            SUM(accident_severity='Fatal')                                 AS Fatal_Accidents,
            SUM(accident_severity='Severe')                                AS Severe_Accidents,
            ROUND(100.0 * SUM(is_fatal_or_severe) / COUNT(*),2)           AS Fatal_Severe_Pct,
            ROUND(100.0 * SUM(adverse_weather)    / COUNT(*),2)           AS Adverse_Weather_Pct,
            ROUND(100.0 * SUM(hazardous_road)     / COUNT(*),2)           AS Hazardous_Road_Pct,
            ROUND(100.0 * SUM(is_peak_hour)       / COUNT(*),2)           AS Peak_Hour_Accident_Pct,
            ROUND(AVG(casualties),2)                                       AS Avg_Casualties_Per_Accident
    FROM    fact_accident
""", 'Accident KPIs')

qdf("""
    SELECT  l.location_name AS Most_Dangerous_Location,
            ROUND(SUM(a.risk_score),2) AS Total_Risk_Score
    FROM    fact_accident a JOIN dim_location l ON a.location_id = l.location_id
    GROUP BY l.location_name ORDER BY Total_Risk_Score DESC LIMIT 1
""", 'Most Dangerous Location')

qdf("""
    SELECT  l.location_name AS Congestion_Hotspot,
            ROUND(AVG(t.congestion_index),4) AS Avg_Congestion_Index
    FROM    fact_traffic t JOIN dim_location l ON t.location_id = l.location_id
    GROUP BY l.location_name ORDER BY Avg_Congestion_Index DESC LIMIT 1
""", 'Top Congestion Hotspot')

qdf("""
    SELECT  v.vehicle_type AS Highest_Risk_Vehicle,
            ROUND(AVG(a.risk_score),3) AS Avg_Risk_Score
    FROM    fact_accident a JOIN dim_vehicle v ON a.vehicle_id = v.vehicle_id
    GROUP BY v.vehicle_type ORDER BY Avg_Risk_Score DESC LIMIT 1
""", 'Highest Risk Vehicle Type')

qdf("""
    SELECT  w.weather_condition AS Most_Dangerous_Weather,
            ROUND(AVG(a.risk_score),3) AS Avg_Risk_Score
    FROM    fact_accident a JOIN dim_weather w ON a.weather_id = w.weather_id
    GROUP BY w.weather_condition ORDER BY Avg_Risk_Score DESC LIMIT 1
""", 'Most Dangerous Weather')

KPI DASHBOARD — MetroCity Smart Traffic System

── Traffic KPIs ──


,Total_Sensor_Readings,Active_Sensors,Total_Vehicles_Recorded,Avg_Congestion_Index,High_Congestion_Pct,Avg_Speed_kmh,Peak_Vehicle_Count
0,300,300,83623.000,0.490,32.000,49.100,494



── Accident KPIs ──


,Total_Accidents,Total_Casualties,Avg_Risk_Score,Max_Risk_Score,Fatal_Accidents,Severe_Accidents,Fatal_Severe_Pct,Adverse_Weather_Pct,Hazardous_Road_Pct,Peak_Hour_Accident_Pct,Avg_Casualties_Per_Accident
0,5000,22467.000,11.039,38.366,1253.000,1237.000,49.800,58.540,50.200,33.300,4.490



── Most Dangerous Location ──


...


── Highest Risk Vehicle Type ──


,Highest_Risk_Vehicle,Avg_Risk_Score
0,Bus,11.345



── Most Dangerous Weather ──


,Most_Dangerous_Weather,Avg_Risk_Score
0,Clear,11.343


,Most_Dangerous_Weather,Avg_Risk_Score
0,Clear,11.343


---
## 🤖 Section 8 — Accident Severity Prediction Model (ML)

In [56]:
# ── Pull enriched data from MySQL for ML ──────────────────────────────────────
# Includes quarter as an additional temporal feature
ml_df = qdf("""
    SELECT  a.accident_id,
            a.location_id,
            l.location_name,
            w.weather_condition,
            v.vehicle_type,
            a.road_condition,
            d.hour, d.is_peak_hour, d.is_weekend, d.month, d.quarter,
            a.number_of_vehicles,
            a.traffic_density_num,
            a.adverse_weather,
            a.hazardous_road,
            a.is_fatal_or_severe
    FROM    fact_accident a
    JOIN    dim_location  l ON a.location_id = l.location_id
    JOIN    dim_date      d ON a.date_id     = d.date_id
    JOIN    dim_vehicle   v ON a.vehicle_id  = v.vehicle_id
    JOIN    dim_weather   w ON a.weather_id  = w.weather_id
""")

print(f'ML dataset : {ml_df.shape[0]:,} rows × {ml_df.shape[1]} columns')
print(f'Target split: {ml_df["is_fatal_or_severe"].value_counts().to_dict()}')
print(f'  → {ml_df["is_fatal_or_severe"].mean()*100:.1f}% are Fatal/Severe')

FEATURES = ['location_name','weather_condition','road_condition','vehicle_type',
            'hour','is_peak_hour','is_weekend','month','quarter',
            'number_of_vehicles','traffic_density_num','adverse_weather','hazardous_road']
TARGET   = 'is_fatal_or_severe'

ml_enc   = ml_df.copy()
encoders = {}
for col in ['location_name','weather_condition','road_condition','vehicle_type']:
    le = LabelEncoder()
    ml_enc[col] = le.fit_transform(ml_enc[col].astype(str))
    encoders[col] = le

X = ml_enc[FEATURES].values
y = ml_enc[TARGET].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain: {len(X_train):,}  |  Test: {len(X_test):,}')
print(' ML data pulled from MySQL.')

ML dataset : 5,000 rows × 16 columns
Target split: {0: 2510, 1: 2490}
  → 49.8% are Fatal/Severe

Train: 4,000  |  Test: 1,000
 ML data pulled from MySQL.


In [64]:
# ── Train & compare 3 models ─────────────────────────────────────────────────
scaler  = StandardScaler()
Xtr_sc  = scaler.fit_transform(X_train)
Xts_sc  = scaler.transform(X_test)
cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

MODELS = {
    'Random Forest'      : (RandomForestClassifier(n_estimators=150, max_depth=10,
                                random_state=42, n_jobs=-1), False),
    'Gradient Boosting'  : (GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                            max_depth=5, random_state=42), False),
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), True),
}

results, trained = [], {}
for name, (model, scaled) in MODELS.items():
    Xtr = Xtr_sc if scaled else X_train
    Xts = Xts_sc if scaled else X_test
    model.fit(Xtr, y_train)
    yp    = model.predict(Xts)
    ypr   = model.predict_proba(Xts)[:,1]
    cv_f1 = cross_val_score(model, Xtr, y_train, cv=cv, scoring='f1', n_jobs=-1)
    results.append({'Model'      : name,
                    'Accuracy'   : round(accuracy_score(y_test,yp),4),
                    'F1_Score'   : round(f1_score(y_test,yp),4),
                    'ROC_AUC'    : round(roc_auc_score(y_test,ypr),4),
                    'CV_F1_Mean' : round(cv_f1.mean(),4),
                    'CV_F1_Std'  : round(cv_f1.std(),4)})
    trained[name] = (model, scaled)
    print(f'\n── {name} ──')
    print(classification_report(y_test, yp, target_names=['Not Fatal/Severe','Fatal/Severe']))

results_df = pd.DataFrame(results)
print('\nModel Comparison:')
display(results_df)

best_name              = results_df.sort_values('ROC_AUC', ascending=False).iloc[0]['Model']
best_model, best_scaled = trained[best_name]
print(f'\nBest Model: {best_name}')


── Random Forest ──
                  precision    recall  f1-score   support

Not Fatal/Severe       0.55      0.61      0.58       502
    Fatal/Severe       0.55      0.49      0.52       498

        accuracy                           0.55      1000
       macro avg       0.55      0.55      0.55      1000
    weighted avg       0.55      0.55      0.55      1000


── Gradient Boosting ──
                  precision    recall  f1-score   support

Not Fatal/Severe       0.51      0.53      0.52       502
    Fatal/Severe       0.51      0.49      0.50       498

        accuracy                           0.51      1000
       macro avg       0.51      0.51      0.51      1000
    weighted avg       0.51      0.51      0.51      1000


── Logistic Regression ──
                  precision    recall  f1-score   support

Not Fatal/Severe       0.52      0.56      0.54       502
    Fatal/Severe       0.52      0.48      0.50       498

        accuracy                           0.52  

,Model,Accuracy,F1_Score,ROC_AUC,CV_F1_Mean,CV_F1_Std
0,Random Forest,0.550,0.519,0.552,0.498,0.020
1,Gradient Boosting,0.512,0.500,0.525,0.485,0.011
2,Logistic Regression,0.518,0.497,0.536,0.500,0.013



Best Model: Random Forest


In [65]:
# ── Feature Importance ───────────────────────────────────────────────────────
rf_model = trained['Random Forest'][0]
feat_imp = pd.DataFrame({
    'Feature'   : FEATURES,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False).round(4)

print(' Feature Importance (Random Forest):')
display(feat_imp)

 Feature Importance (Random Forest):


,Feature,Importance
4,hour,0.196
0,location_name,0.108
3,vehicle_type,0.106
7,month,0.100
9,number_of_vehicles,0.090
1,weather_condition,0.089
10,traffic_density_num,0.071
2,road_condition,0.071
6,is_weekend,0.038
8,quarter,0.036


In [59]:
# ── LOAD: ml_accident_risk — write predictions directly into MySQL ────────────
X_full    = scaler.transform(X) if best_scaled else X
pred_prob = best_model.predict_proba(X_full)[:,1].round(4)
pred_cls  = best_model.predict(X_full)

def risk_tier(p):
    if   p <= 0.30: return 'Low Risk'
    elif p <= 0.50: return 'Moderate Risk'
    elif p <= 0.70: return 'High Risk'
    else:           return 'Critical Risk'

ml_rows = [
    (ml_df.iloc[i]['accident_id'],
     int(ml_df.iloc[i]['location_id']),
     ml_df.iloc[i]['location_name'],
     float(pred_prob[i]),
     int(pred_cls[i]),
     risk_tier(pred_prob[i]),
     int(y[i]))
    for i in range(len(ml_df))
]

conn = get_conn(); cur = conn.cursor()
exec_sql(cur,
    '''INSERT IGNORE INTO ml_accident_risk
       (accident_id, location_id, location_name, predicted_risk_prob,
        predicted_fatal_severe, risk_tier, actual_fatal_severe)
       VALUES (%s,%s,%s,%s,%s,%s,%s)''',
    ml_rows, many=True)
conn.commit(); conn.close()

print(f'✅ ml_accident_risk → {len(ml_rows):,} rows')
qdf("""
    SELECT  risk_tier, COUNT(*) AS Count,
            ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER(),1) AS Pct
    FROM    ml_accident_risk
    GROUP BY risk_tier ORDER BY Count DESC
""", 'Risk Tier Distribution')

✅ ml_accident_risk → 5,000 rows

── Risk Tier Distribution ──


,risk_tier,Count,Pct
0,Moderate Risk,2336,46.700
1,High Risk,2112,42.200
2,Critical Risk,297,5.900
3,Low Risk,255,5.100


,risk_tier,Count,Pct
0,Moderate Risk,2336,46.700
1,High Risk,2112,42.200
2,Critical Risk,297,5.900
3,Low Risk,255,5.100


---
## 🔭 Section 9 — MySQL Views (Power BI Ready)

In [60]:
# ─────────────────────────────────────────────────────────────────────────────
# 8 MySQL VIEWS — pre-aggregated, named, ready to import directly into Power BI.
# Each view maps to one specific Power BI dashboard visual or page.
# Updating the underlying fact tables automatically refreshes all views.
# ─────────────────────────────────────────────────────────────────────────────

VIEW_STMTS = [

    'DROP VIEW IF EXISTS vw_ml_location_risk',
    'DROP VIEW IF EXISTS vw_road_condition_risk',
    'DROP VIEW IF EXISTS vw_severity_by_location',
    'DROP VIEW IF EXISTS vw_weather_risk',
    'DROP VIEW IF EXISTS vw_congestion_hotspots',
    'DROP VIEW IF EXISTS vw_monthly_trend',
    'DROP VIEW IF EXISTS vw_hourly_profile',
    'DROP VIEW IF EXISTS vw_location_summary',

    # ── vw_location_summary ───────────────────────────────────────────────────
    # ONE ROW PER LOCATION. Powers: location bar chart, KPI cards per location,
    # combined traffic + accident overview.
    # ADDED: accident_rate_per_1000_vehicles — standard traffic safety KPI
    #        = (total_accidents / total_vehicles) × 1000
    '''
    CREATE VIEW vw_location_summary AS
    SELECT
        l.location_id,
        l.location_name,
        l.zone_type,
        COUNT(DISTINCT t.traffic_id)                                AS sensor_readings,
        COUNT(DISTINCT t.sensor_id)                                 AS active_sensors,
        ROUND(AVG(t.vehicle_count),1)                               AS avg_vehicle_count,
        MAX(t.vehicle_count)                                        AS peak_vehicle_count,
        COALESCE(SUM(t.vehicle_count),0)                            AS total_vehicles,
        ROUND(AVG(t.average_speed),1)                               AS avg_speed_kmh,
        ROUND(AVG(t.congestion_index),4)                            AS avg_congestion_index,
        ROUND(100.0*SUM(t.congestion_level='High')/
              NULLIF(COUNT(t.traffic_id),0),1)                      AS high_congestion_pct,
        COUNT(DISTINCT a.accident_pk)                               AS total_accidents,
        COALESCE(SUM(a.casualties),0)                               AS total_casualties,
        ROUND(COALESCE(AVG(a.risk_score),0),3)                      AS avg_risk_score,
        ROUND(COALESCE(SUM(a.risk_score),0),2)                      AS total_risk_score,
        COALESCE(SUM(a.accident_severity='Fatal'),0)                AS fatal_accidents,
        COALESCE(SUM(a.accident_severity='Severe'),0)               AS severe_accidents,
        ROUND(100.0*COALESCE(SUM(a.is_fatal_or_severe),0)/
              NULLIF(COUNT(DISTINCT a.accident_pk),0),1)            AS fatal_severe_pct,
        ROUND(1000.0 * COUNT(DISTINCT a.accident_pk) /
              NULLIF(COALESCE(SUM(t.vehicle_count),0),0), 4)        AS accident_rate_per_1000_vehicles
    FROM      dim_location  l
    LEFT JOIN fact_traffic   t ON l.location_id = t.location_id
    LEFT JOIN fact_accident  a ON l.location_id = a.location_id
    GROUP BY  l.location_id, l.location_name, l.zone_type
    ''',

    # ── vw_hourly_profile ──────────────────────────────────────────────────────
    # ONE ROW PER HOUR (0–23). Powers: 24-hour line chart, peak hour analysis bar.
    '''
    CREATE VIEW vw_hourly_profile AS
    SELECT
        d.hour,
        d.time_of_day,
        d.is_peak_hour,
        ROUND(AVG(t.vehicle_count),1)                               AS avg_vehicles,
        ROUND(AVG(t.average_speed),1)                               AS avg_speed,
        ROUND(AVG(t.congestion_index),4)                            AS avg_congestion_index,
        SUM(t.congestion_level='High')                              AS high_congestion_count,
        COUNT(DISTINCT a.accident_pk)                               AS accident_count,
        COALESCE(SUM(a.casualties),0)                               AS total_casualties,
        ROUND(COALESCE(AVG(a.risk_score),0),3)                      AS avg_risk_score,
        COALESCE(SUM(a.accident_severity='Fatal'),0)                AS fatal_count
    FROM      dim_date     d
    LEFT JOIN fact_traffic  t ON d.date_id = t.date_id
    LEFT JOIN fact_accident a ON d.date_id = a.date_id
    GROUP BY  d.hour, d.time_of_day, d.is_peak_hour
    ORDER BY  d.hour
    ''',

    # ── vw_monthly_trend ──────────────────────────────────────────────────────
    # ONE ROW PER YEAR-MONTH. Powers: monthly trend line, quarter comparison chart.
    # month_sort ensures Power BI sorts January–December correctly.
    '''
    CREATE VIEW vw_monthly_trend AS
    SELECT
        d.year,
        d.quarter,
        d.month        AS month_num,
        d.month_name,
        d.month_sort,
        COUNT(DISTINCT a.accident_pk)                               AS total_accidents,
        COALESCE(SUM(a.casualties),0)                               AS total_casualties,
        ROUND(COALESCE(AVG(a.risk_score),0),3)                      AS avg_risk_score,
        COALESCE(SUM(a.accident_severity='Fatal'),0)                AS fatal_accidents,
        COALESCE(SUM(a.accident_severity='Severe'),0)               AS severe_accidents,
        COALESCE(SUM(a.is_fatal_or_severe),0)                       AS fatal_severe_total,
        COALESCE(SUM(a.adverse_weather),0)                          AS adverse_weather_accidents
    FROM      dim_date     d
    LEFT JOIN fact_accident a ON d.date_id = a.date_id
    GROUP BY  d.year, d.quarter, d.month, d.month_name, d.month_sort
    ORDER BY  d.year, d.month
    ''',

    # ── vw_congestion_hotspots ────────────────────────────────────────────────
    # LOCATION × TIME PERIOD rows. Powers: heatmap (location vs time period),
    # congestion hotspot ranked bar chart.
    '''
    CREATE VIEW vw_congestion_hotspots AS
    SELECT
        l.location_name,
        l.zone_type,
        d.time_of_day,
        d.is_peak_hour,
        COUNT(*)                                                    AS readings,
        ROUND(AVG(t.vehicle_count),1)                               AS avg_vehicles,
        ROUND(AVG(t.congestion_index),4)                            AS avg_congestion_index,
        SUM(t.congestion_level='High')                              AS high_congestion_hits,
        ROUND(100.0*SUM(t.congestion_level='High')/COUNT(*),1)      AS high_congestion_pct
    FROM    fact_traffic t
    JOIN    dim_location l ON t.location_id = l.location_id
    JOIN    dim_date     d ON t.date_id     = d.date_id
    GROUP BY l.location_name, l.zone_type, d.time_of_day, d.is_peak_hour
    ORDER BY avg_congestion_index DESC
    ''',

    # ── vw_weather_risk ───────────────────────────────────────────────────────
    # WEATHER × ROAD CONDITION matrix. Powers: matrix visual / stacked bar.
    '''
    CREATE VIEW vw_weather_risk AS
    SELECT
        w.weather_condition,
        w.is_adverse                                                AS is_adverse_weather,
        a.road_condition,
        a.hazardous_road                                            AS is_hazardous_road,
        COUNT(*)                                                    AS accidents,
        SUM(a.casualties)                                           AS total_casualties,
        ROUND(AVG(a.severity_score),2)                              AS avg_severity_score,
        ROUND(AVG(a.risk_score),3)                                  AS avg_risk_score,
        SUM(a.accident_severity='Fatal')                            AS fatal_count,
        ROUND(100.0*SUM(a.is_fatal_or_severe)/COUNT(*),1)           AS fatal_severe_pct
    FROM    fact_accident a
    JOIN    dim_weather   w ON a.weather_id = w.weather_id
    GROUP BY w.weather_condition, w.is_adverse, a.road_condition, a.hazardous_road
    ORDER BY avg_risk_score DESC
    ''',

    # ── vw_severity_by_location ───────────────────────────────────────────────
    # LOCATION × SEVERITY LEVEL rows. Powers: 100% stacked bar chart showing
    # the breakdown of Minor/Moderate/Severe/Fatal accidents per location.
    # severity_sort ensures the chart stacks in the correct order.
    '''
    CREATE VIEW vw_severity_by_location AS
    SELECT
        l.location_name,
        l.zone_type,
        a.accident_severity,
        a.severity_score                                            AS severity_sort,
        COUNT(*)                                                    AS accident_count,
        SUM(a.casualties)                                           AS total_casualties,
        ROUND(AVG(a.risk_score),3)                                  AS avg_risk_score,
        ROUND(100.0 * COUNT(*) /
              SUM(COUNT(*)) OVER (PARTITION BY l.location_name),1)  AS pct_of_location
    FROM    fact_accident a
    JOIN    dim_location  l ON a.location_id = l.location_id
    GROUP BY l.location_name, l.zone_type, a.accident_severity, a.severity_score
    ORDER BY l.location_name, severity_sort DESC
    ''',

    # ── vw_road_condition_risk ────────────────────────────────────────────────
    # STANDALONE ROAD CONDITION BREAKDOWN. Powers: dedicated road condition bar chart.
    # Required by the case study — road condition is a core analysis dimension.
    '''
    CREATE VIEW vw_road_condition_risk AS
    SELECT
        a.road_condition,
        a.hazardous_road                                            AS is_hazardous,
        COUNT(*)                                                    AS accidents,
        SUM(a.casualties)                                           AS total_casualties,
        ROUND(AVG(a.severity_score),2)                              AS avg_severity,
        ROUND(AVG(a.risk_score),3)                                  AS avg_risk_score,
        SUM(a.accident_severity='Fatal')                            AS fatal_count,
        ROUND(100.0*SUM(a.is_fatal_or_severe)/COUNT(*),1)           AS fatal_severe_pct
    FROM    fact_accident a
    GROUP BY a.road_condition, a.hazardous_road
    ORDER BY avg_risk_score DESC
    ''',

    # ── vw_ml_location_risk ───────────────────────────────────────────────────
    # ONE ROW PER LOCATION — aggregated ML predictions.
    # Powers: Power BI MAP visual where each location bubble is coloured by
    # avg predicted risk probability (replace with lat/lon in Power BI map settings).
    # Also powers: risk tier donut per location, safety rating table.
    '''
    CREATE VIEW vw_ml_location_risk AS
    SELECT
        m.location_id,
        m.location_name,
        l.zone_type,
        COUNT(*)                                                    AS total_predictions,
        ROUND(AVG(m.predicted_risk_prob),4)                         AS avg_predicted_risk,
        ROUND(MAX(m.predicted_risk_prob),4)                         AS max_predicted_risk,
        SUM(m.predicted_fatal_severe)                               AS predicted_fatal_severe_count,
        SUM(m.risk_tier = 'Critical Risk')                          AS critical_risk_count,
        SUM(m.risk_tier = 'High Risk')                              AS high_risk_count,
        SUM(m.risk_tier = 'Moderate Risk')                          AS moderate_risk_count,
        SUM(m.risk_tier = 'Low Risk')                               AS low_risk_count,
        ROUND(100.0 * SUM(m.actual_fatal_severe) / COUNT(*),1)      AS actual_fatal_severe_pct,
        ROUND(100.0 * SUM(m.predicted_fatal_severe) / COUNT(*),1)   AS predicted_fatal_severe_pct,
        CASE
            WHEN AVG(m.predicted_risk_prob) > 0.70 THEN 'Very Dangerous'
            WHEN AVG(m.predicted_risk_prob) > 0.50 THEN 'Dangerous'
            WHEN AVG(m.predicted_risk_prob) > 0.30 THEN 'Caution'
            ELSE                                        'Safe'
        END                                                         AS safety_rating
    FROM      ml_accident_risk m
    JOIN      dim_location     l ON m.location_id = l.location_id
    GROUP BY  m.location_id, m.location_name, l.zone_type
    ORDER BY  avg_predicted_risk DESC
    '''
]

conn = get_conn(); cur = conn.cursor()
for s in VIEW_STMTS:
    exec_sql(cur, s.strip())
conn.commit(); conn.close()

print('✅ Power BI Views created:')
for v in ['vw_location_summary','vw_hourly_profile','vw_monthly_trend',
          'vw_congestion_hotspots','vw_weather_risk','vw_severity_by_location',
          'vw_road_condition_risk','vw_ml_location_risk']:
    print(f'   • {v}')

✅ Power BI Views created:
   • vw_location_summary
   • vw_hourly_profile
   • vw_monthly_trend
   • vw_congestion_hotspots
   • vw_weather_risk
   • vw_severity_by_location
   • vw_road_condition_risk
   • vw_ml_location_risk


In [61]:
# ── Preview all 8 views ───────────────────────────────────────────────────────
for view, label in [
    ('vw_location_summary'  , '📍 Location Summary (incl. accident_rate_per_1000_vehicles)'),
    ('vw_hourly_profile'    , '⏰ Hourly Profile'),
    ('vw_monthly_trend'     , '📅 Monthly Trend'),
    ('vw_congestion_hotspots','🚦 Congestion Hotspots'),
    ('vw_weather_risk'      , '🌧️ Weather Risk Matrix'),
    ('vw_severity_by_location','📊 Severity Breakdown by Location'),
    ('vw_road_condition_risk', '🛣️ Road Condition Risk'),
    ('vw_ml_location_risk'  , '🤖 ML Location Risk (map-ready)'),
]:
    qdf(f'SELECT * FROM {view} LIMIT 5', label)


── 📍 Location Summary (incl. accident_rate_per_1000_vehicles) ──


,location_id,location_name,zone_type,sensor_readings,active_sensors,avg_vehicle_count,peak_vehicle_count,total_vehicles,avg_speed_kmh,avg_congestion_index,high_congestion_pct,total_accidents,total_casualties,avg_risk_score,total_risk_score,fatal_accidents,severe_accidents,fatal_severe_pct,accident_rate_per_1000_vehicles
0,1,Downtown,Urban Core,55,55,290.600,492,15565494.000,49.400,0.503,32.700,974,242275.000,10.777,577316.190,12210.000,12815.000,2569.300,0.063
1,2,Highway,Highway,80,80,257.800,491,20994214.000,49.700,0.461,32.500,1018,373280.000,11.404,928702.090,20800.000,19520.000,3960.700,0.049
2,3,Industrial Area,Industrial,51,51,283.800,494,14371689.000,50.200,0.491,25.500,993,227715.000,11.234,568943.590,12903.000,14025.000,2711.800,0.069
3,4,Residential Zone,Residential,59,59,329.400,494,19397128.000,45.500,0.570,23.700,998,262609.000,11.205,659792.060,14337.000,15340.000,2973.600,0.051
4,5,Suburbs,Suburban,55,55,238.400,477,13332870.000,50.800,0.432,45.500,1017,246400.000,10.569,591185.400,15125.000,12375.000,2704.000,0.076



── ⏰ Hourly Profile ──


,hour,time_of_day,is_peak_hour,avg_vehicles,avg_speed,avg_congestion_index,high_congestion_count,accident_count,total_casualties,avg_risk_score,fatal_count
0,0,Off-Peak Night,0,243.000,49.500,0.444,3.000,209,940.000,10.748,50.000
1,1,Off-Peak Night,0,286.200,50.700,0.491,6.000,209,937.000,11.250,58.000
2,2,Off-Peak Night,0,312.500,48.400,0.535,4.000,209,898.000,10.319,46.000
3,3,Off-Peak Night,0,269.400,41.300,0.518,4.000,209,970.000,10.975,44.000
4,4,Off-Peak Night,0,305.700,53.600,0.500,5.000,209,932.000,10.576,46.000



── 📅 Monthly Trend ──


...

,location_name,zone_type,accident_severity,severity_sort,accident_count,total_casualties,avg_risk_score,pct_of_location
0,Downtown,Urban Core,Fatal,4,222,973.000,18.017,22.800
1,Downtown,Urban Core,Severe,3,233,997.000,13.187,23.900
2,Downtown,Urban Core,Moderate,2,258,1219.000,8.881,26.500
3,Downtown,Urban Core,Minor,1,261,1216.000,4.341,26.800
4,Highway,Highway,Fatal,4,260,1175.000,17.815,25.500



── 🛣️ Road Condition Risk ──


,road_condition,is_hazardous,accidents,total_casualties,avg_severity,avg_risk_score,fatal_count,fatal_severe_pct
0,Dry,0,1257,5657.000,2.540,11.390,329.000,52.000
1,Icy,1,1288,5887.000,2.460,11.040,301.000,48.600
2,Under Construction,1,1222,5420.000,2.510,10.996,312.000,50.200
3,Wet,0,1233,5503.000,2.470,10.721,311.000,48.300



── 🤖 ML Location Risk (map-ready) ──


,location_id,location_name,zone_type,total_predictions,avg_predicted_risk,max_predicted_risk,predicted_fatal_severe_count,critical_risk_count,high_risk_count,moderate_risk_count,low_risk_count,actual_fatal_severe_pct,predicted_fatal_severe_pct,safety_rating
0,3,Industrial Area,Industrial,993,0.518,0.878,539.000,54.000,485.000,440.000,14.000,53.200,54.300,Dangerous
1,4,Residential Zone,Residential,998,0.504,0.851,482.000,64.000,418.000,489.000,27.000,50.400,48.300,Dangerous
2,5,Suburbs,Suburban,1017,0.495,0.844,493.000,63.000,430.000,465.000,59.000,49.200,48.500,Caution
3,2,Highway,Highway,1018,0.488,0.838,451.000,53.000,398.000,506.000,61.000,49.500,44.300,Caution
4,1,Downtown,Urban Core,974,0.484,0.859,444.000,63.000,381.000,436.000,94.000,46.700,45.600,Caution


---
## ✅ Section 10 — Verification

In [62]:
conn = get_conn(); cur = conn.cursor()
print('=' * 52)
print('MySQL Object Verification — smart_city_dw')
print('=' * 52)

TABLES_LIST = ['dim_location','dim_date','dim_vehicle','dim_weather',
               'fact_traffic','fact_accident','ml_accident_risk']
VIEWS_LIST  = ['vw_location_summary','vw_hourly_profile','vw_monthly_trend',
               'vw_congestion_hotspots','vw_weather_risk',
               'vw_severity_by_location','vw_road_condition_risk','vw_ml_location_risk']

print('\nTables:')
for t in TABLES_LIST:
    cur.execute(f'SELECT COUNT(*) FROM {t}')
    print(f'  {t:<25}  {cur.fetchone()[0]:>7,} rows')

print('\nViews (row counts):')
for v in VIEWS_LIST:
    cur.execute(f'SELECT COUNT(*) FROM {v}')
    print(f'  {v:<30}  {cur.fetchone()[0]:>7,} rows')

conn.close()
print('\n' + '=' * 52)
print('✅ smart_city_dw is fully ready for Power BI.')

MySQL Object Verification — smart_city_dw

Tables:
  dim_location                     5 rows
  dim_date                     5,000 rows
  dim_vehicle                      5 rows
  dim_weather                      5 rows
  fact_traffic                   300 rows
  fact_accident                5,000 rows
  ml_accident_risk             5,000 rows

Views (row counts):
  vw_location_summary                   5 rows
  vw_hourly_profile                    24 rows
  vw_monthly_trend                      7 rows
  vw_congestion_hotspots               25 rows
  vw_weather_risk                      20 rows
  vw_severity_by_location              20 rows
  vw_road_condition_risk                4 rows
  vw_ml_location_risk                   5 rows

✅ smart_city_dw is fully ready for Power BI.


---
## 📊 Power BI Connection & Dashboard Guide

### Step 1 — Connect Power BI to MySQL
```
Power BI Desktop → Home → Get Data → MySQL Database
Server   : localhost
Database : smart_city_dw
```
*(Install MySQL Connector/NET 8.x if prompted)*

### Step 2 — Import Objects
Import all 7 **tables** + all 8 **views**. The views are pre-aggregated and load faster than running measures on raw facts.

### Step 3 — Relationships (Model View)
```
dim_location.location_id  ──<  fact_traffic.location_id
dim_location.location_id  ──<  fact_accident.location_id
dim_location.location_id  ──<  ml_accident_risk.location_id
dim_date.date_id           ──<  fact_traffic.date_id
dim_date.date_id           ──<  fact_accident.date_id
dim_vehicle.vehicle_id     ──<  fact_accident.vehicle_id
dim_weather.weather_id     ──<  fact_accident.weather_id
```

### Step 4 — Sort Columns (Power BI Column Tools)
| Column | Sort By |
|--------|---------|
| `dim_date.month_name` | `dim_date.month_sort` |
| `dim_date.day_name` | `dim_date.day_of_week_num` |
| `vw_monthly_trend.month_name` | `vw_monthly_trend.month_sort` |

### Suggested Dashboards & Which Objects to Use
| Dashboard Page | Primary Objects | Key Columns |
|----------------|----------------|-------------|
| **Traffic Overview** | `vw_location_summary`, `vw_hourly_profile` | `avg_congestion_index`, `avg_vehicle_count`, `high_congestion_pct` |
| **Peak Hour Analysis** | `vw_congestion_hotspots`, `vw_hourly_profile` | `time_of_day`, `avg_ci`, `high_congestion_pct` |
| **Accident Analytics** | `vw_location_summary`, `vw_severity_by_location` | `total_risk_score`, `fatal_severe_pct`, `total_casualties` |
| **Weather & Road Safety** | `vw_weather_risk`, `vw_road_condition_risk` | `avg_risk_score`, `fatal_count`, `is_adverse_weather` |
| **Monthly Trends** | `vw_monthly_trend` | `total_accidents`, `fatal_accidents`, `quarter` |
| **Predictive Risk Map** | `vw_ml_location_risk` | `avg_predicted_risk`, `safety_rating`, `critical_risk_count` |
| **KPI Summary Page** | `vw_location_summary` | `accident_rate_per_1000_vehicles`, `avg_risk_score` |